In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

# =============================================================================
# 1. CARGA DE DATOS Y CONFIGURACIÓN ÓPTIMA
# =============================================================================
train_df = pd.read_csv('train_selected_RandomForest.csv')
tuning_df = pd.read_csv('tuning_selected_RandomForest.csv')

target_col = 'Class'
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_tuning = tuning_df.drop(columns=[target_col])
y_tuning = tuning_df[target_col]

le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_tuning_num = le.transform(y_tuning)

# =============================================================================
# 2. DEFINICIÓN DE MODELOS CON LOS MEJORES PARÁMETROS (OPTIMIZADOS)
# =============================================================================
modelos = {
    'Regresión Logística': LogisticRegression(
        C=0.1, 
        penalty='l1', 
        solver='saga', 
        random_state=42
    ),
    'SVM': SVC(
        C=100, 
        gamma=0.01, 
        kernel='rbf', 
        random_state=42
    ),
    'k-NN': KNeighborsClassifier(
        n_neighbors=5, 
        p=1, 
        weights='distance'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, 
        min_samples_split=2, 
        min_samples_leaf=2, 
        max_depth=None, 
        criterion='entropy', 
        random_state=42
    )
}

# Diccionario para almacenar las predicciones de cada modelo en Tuning
preds = {}

print("Entrenando modelos optimizados y obteniendo predicciones sobre Tuning...")
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train_num)
    preds[nombre] = modelo.predict(X_tuning)

# =============================================================================
# 3. CÁLCULO DE MÉTRICAS DE DIVERSIDAD (Parejas de Modelos)
# =============================================================================
# Generamos todas las combinaciones posibles de pares de modelos (6 parejas en total)
pares = list(combinations(modelos.keys(), 2))
N = len(y_tuning_num)

resultados_diversidad = []

for mod1, mod2 in pares:
    pred1 = preds[mod1]
    pred2 = preds[mod2]
    
    # Tabla de contingencia matemática
    # Ambos aciertan (N11), Mod1 acierta y Mod2 falla (N10)...
    n11 = np.sum((pred1 == y_tuning_num) & (pred2 == y_tuning_num))
    n10 = np.sum((pred1 == y_tuning_num) & (pred2 != y_tuning_num))
    n01 = np.sum((pred1 != y_tuning_num) & (pred2 == y_tuning_num))
    n00 = np.sum((pred1 != y_tuning_num) & (pred2 != y_tuning_num)) # Ambos fallan
    
    # 1. Métrica de Desacuerdo (Disagreement Measure)
    # Proporción de casos donde un modelo acierta y el otro falla
    desacuerdo = (n10 + n01) / N
    
    # 2. Métrica de Doble Fallo (Double Fault Measure)
    # Proporción de casos donde AMBOS modelos se equivocan simultáneamente
    doble_fallo = n00 / N
    
    resultados_diversidad.append({
        'Modelo A': mod1,
        'Modelo B': mod2,
        'Desacuerdo ↑': desacuerdo,
        'Doble Fallo ↓': doble_fallo
    })


df_diversidad = pd.DataFrame(resultados_diversidad)

print("\n" + "="*70)
print("MATRIZ DE DIVERSIDAD ENTRE MODELOS OPTIMIZADOS (Sobre Tuning Set)")
print("="*70)
print(df_diversidad.to_string(index=False))
print("="*70)

Entrenando modelos optimizados y obteniendo predicciones sobre Tuning...

📊 MATRIZ DE DIVERSIDAD ENTRE MODELOS OPTIMIZADOS (Sobre Tuning Set)
           Modelo A      Modelo B  Desacuerdo ↑  Doble Fallo ↓
Regresión Logística           SVM      0.064516       0.161290
Regresión Logística          k-NN      0.145161       0.096774
Regresión Logística Random Forest      0.096774       0.112903
                SVM          k-NN      0.080645       0.129032
                SVM Random Forest      0.129032       0.096774
               k-NN Random Forest      0.048387       0.112903
